In [ ]:
from io import BytesIO
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import tifffile
from IPython.display import display
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
VOLUME_PATH = ROOT / "output" / "reference_generate_smoke" / "volume_000" / "volume.tiff"

In [ ]:
volume = tifffile.imread(VOLUME_PATH).astype(np.float32)
print("volume:", VOLUME_PATH)
print("shape:", volume.shape)
print("values:", np.unique(volume))

In [ ]:
PALETTE = {
    0.0: (45, 45, 45),
    0.5: (155, 155, 155),
    1.0: (245, 245, 245),
}


def slice_png_bytes(slice_2d, scale=8):
    rgb = np.empty((*slice_2d.shape, 3), dtype=np.uint8)
    for value, color in PALETTE.items():
        rgb[np.isclose(slice_2d, value)] = color

    image = Image.fromarray(rgb, mode="RGB")
    if scale != 1:
        image = image.resize((image.width * scale, image.height * scale), Image.Resampling.NEAREST)

    buffer = BytesIO()
    image.save(buffer, format="PNG")
    return buffer.getvalue()


frames = [slice_png_bytes(volume[z]) for z in range(volume.shape[0])]
slider = widgets.IntSlider(
    value=volume.shape[0] // 2,
    min=0,
    max=volume.shape[0] - 1,
    step=1,
    description="z",
    continuous_update=True,
    layout=widgets.Layout(width="640px"),
)
label = widgets.Label()
image = widgets.Image(value=frames[slider.value], format="png")


def update(index):
    image.value = frames[index]
    label.value = f"z = {index} / {volume.shape[0] - 1}"


def on_change(change):
    update(change["new"])


slider.observe(on_change, names="value")
update(slider.value)
display(widgets.VBox([slider, label, image]))